# 01 - Data Preprocessing Pipeline
## Mục tiêu
Làm sạch và chuẩn hóa 2.6 triệu giao dịch từ file `kz.csv` trước khi đưa vào phân tích RFM và Streaming.

### Các bước thực hiện:
1. Đọc dữ liệu bằng Spark DataFrame  
2. Xử lý Missing Values  
3. Xử lý Duplicate  
4. Chuyển đổi kiểu dữ liệu  
5. Tạo cột mới (`total_amount`, `date`)  
6. Chuẩn hóa chuỗi (`brand`, `category`)  
7. Spark ML Pipeline (StringIndexer → VectorAssembler → StandardScaler)  
8. Lưu kết quả dạng Parquet

## 1. Import thư viện & Khởi tạo SparkSession

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    DoubleType, TimestampType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler, StandardScaler
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (SparkSession.builder
         .appName("BCCK_Preprocessing")
         .config("spark.sql.shuffle.partitions", "200")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")


Spark version : 3.5.0
Spark UI      : http://99acca626fe5:4040


## 2. Đọc dữ liệu CSV bằng Spark DataFrame

In [2]:
# ── Định nghĩa schema tường minh ────────────────────────────────────────────
schema = StructType([
    StructField("event_time",    StringType(),  True),
    StructField("order_id",      StringType(),  True),
    StructField("product_id",    StringType(),  True),
    StructField("category_id",   StringType(),  True),
    StructField("category_code", StringType(),  True),
    StructField("brand",         StringType(),  True),
    StructField("price",         DoubleType(),  True),
    StructField("user_id",       StringType(),  True),
])

DATA_PATH = "/home/jovyan/work/data/kz.csv"          # chỉnh lại nếu chạy ở thư mục khác
OUTPUT_PATH = "/home/jovyan/work/data/cleaned_orders.parquet"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .schema(schema)
          .csv(DATA_PATH))

total_raw = df_raw.count()
print(f"Tổng số giao dịch (raw)  : {total_raw:,}")
print(f"Số cột                   : {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(5, truncate=False)


Tổng số giao dịch (raw)  : 2,633,521
Số cột                   : 8
root
 |-- event_time: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)

+-----------------------+-------------------+-------------------+-------------------+---------------------------+-------+------+-------------------+
|event_time             |order_id           |product_id         |category_id        |category_code              |brand  |price |user_id            |
+-----------------------+-------------------+-------------------+-------------------+---------------------------+-------+------+-------------------+
|2020-04-24 11:50:39 UTC|2294359932054536986|1515966223509089906|2268105426648170900|electronics.tablet         |samsung|162.01|1515915625441993984|
|2020-0

## 3. Phân tích Missing Values

> Trước khi xử lý, ta visualize tỷ lệ null cho từng cột.

In [3]:
# ── Đếm null từng cột ────────────────────────────────────────────────────────
null_counts = {col: df_raw.filter(F.col(col).isNull()).count()
               for col in df_raw.columns}
null_df = pd.DataFrame(list(null_counts.items()),
                       columns=["column", "null_count"])
null_df["null_pct"] = (null_df["null_count"] / total_raw * 100).round(2)
null_df = null_df.sort_values("null_pct", ascending=False)
print(null_df.to_string(index=False))

# ── Biểu đồ TRƯỚC khi xử lý ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Missing Values - TRƯỚC khi xử lý", fontsize=14, fontweight="bold")

ax = axes[0]
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in null_df["null_count"]]
bars = ax.barh(null_df["column"], null_df["null_count"], color=colors)
ax.set_xlabel("Số lượng null")
ax.set_title("Số lượng giá trị null theo cột")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, null_df["null_count"]):
    if val > 0:
        ax.text(bar.get_width() + total_raw*0.001, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", fontsize=9)

ax2 = axes[1]
ax2.barh(null_df["column"], null_df["null_pct"], color=colors)
ax2.set_xlabel("Tỷ lệ null (%)")
ax2.set_title("Tỷ lệ null (%) theo cột")
for bar, val in zip(ax2.patches, null_df["null_pct"]):
    if val > 0:
        ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("missing_before.png", dpi=120, bbox_inches="tight")
plt.show()
print("\nĐã lưu: missing_before.png")


       column  null_count  null_pct
      user_id     2069352     78.58
category_code      612202     23.25
        brand      506005     19.21
  category_id      431954     16.40
        price      431954     16.40
   event_time           0      0.00
     order_id           0      0.00
   product_id           0      0.00

Đã lưu: missing_before.png


## 4. Xử lý Missing Values

In [4]:
# ── Chiến lược xử lý null ────────────────────────────────────────────────────
# price     → xóa dòng (không có giá không có ý nghĩa giao dịch)
# user_id   → xóa dòng (không có user không thể tính RFM)
# brand     → điền "unknown"
# category_code → điền "unknown"
# category_id   → điền "0"

df_clean = (df_raw
    .dropna(subset=["price", "user_id"])
    .fillna({"brand": "unknown",
             "category_code": "unknown",
             "category_id": "0"}))

removed_missing = total_raw - df_clean.count()
print(f"Đã xóa {removed_missing:,} dòng do thiếu price/user_id")
print(f"Còn lại sau xử lý null: {df_clean.count():,} dòng")


Đã xóa 2,069,352 dòng do thiếu price/user_id
Còn lại sau xử lý null: 564,169 dòng


## 5. Xử lý Duplicate

In [5]:
cnt_before_dedup = df_clean.count()

# Các cột khóa định danh duy nhất một giao dịch
KEY_COLS = ["event_time", "order_id", "product_id", "user_id"]
df_dedup = df_clean.dropDuplicates(KEY_COLS)

cnt_after_dedup = df_dedup.count()
removed_dup = cnt_before_dedup - cnt_after_dedup
dup_pct = removed_dup / cnt_before_dedup * 100

print(f"Trước khi dedup : {cnt_before_dedup:,}")
print(f"Sau khi dedup   : {cnt_after_dedup:,}")
print(f"Đã xóa          : {removed_dup:,} dòng ({dup_pct:.2f}%)")

# Mini bar chart
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Raw (sau null-drop)", "Sau khi dedup"]
values = [cnt_before_dedup, cnt_after_dedup]
bars = ax.bar(labels, values, color=["#3498db", "#2ecc71"], width=0.4)
ax.set_title("Số dòng trước và sau khi loại Duplicate", fontweight="bold")
ax.set_ylabel("Số giao dịch")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + cnt_before_dedup*0.005,
            f"{val:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig("dedup_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


Trước khi dedup : 564,169
Sau khi dedup   : 563,495
Đã xóa          : 674 dòng (0.12%)


## 6. Chuyển đổi kiểu dữ liệu & Tạo cột mới

In [6]:
# ── Chuyển đổi kiểu dữ liệu ──────────────────────────────────────────────────
df_typed = (df_dedup
    # event_time: "2020-04-24 11:50:39 UTC" → timestamp
    .withColumn("event_time",
                F.to_timestamp(F.regexp_replace("event_time", " UTC$", ""),
                               "yyyy-MM-dd HH:mm:ss"))
    # price → double (đã khai báo schema, cast lại để chắc chắn)
    .withColumn("price", F.col("price").cast(DoubleType()))
    # Chuẩn hóa string
    .withColumn("user_id",    F.col("user_id").cast(StringType()))
    .withColumn("order_id",   F.col("order_id").cast(StringType()))
    .withColumn("product_id", F.col("product_id").cast(StringType()))
)

# ── Tạo cột mới ───────────────────────────────────────────────────────────────
# quantity: dataset này không có sẵn → giả định mỗi dòng = 1 đơn vị
df_typed = (df_typed
    .withColumn("quantity", F.lit(1).cast("int"))
    .withColumn("total_amount", F.col("price") * F.col("quantity"))
    .withColumn("date", F.date_trunc("day", F.col("event_time")).cast("date"))
    .withColumn("year_month",
                F.date_format(F.col("event_time"), "yyyy-MM"))
)

print("Schema sau khi chuyển đổi:")
df_typed.printSchema()
df_typed.select("event_time", "price", "quantity",
                "total_amount", "date", "year_month").show(5)


Schema sau khi chuyển đổi:
root
 |-- event_time: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category_id: string (nullable = false)
 |-- category_code: string (nullable = false)
 |-- brand: string (nullable = false)
 |-- price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- quantity: integer (nullable = false)
 |-- total_amount: double (nullable = true)
 |-- date: date (nullable = true)
 |-- year_month: string (nullable = true)

+-------------------+-----+--------+------------+----------+----------+
|         event_time|price|quantity|total_amount|      date|year_month|
+-------------------+-----+--------+------------+----------+----------+
|2020-04-29 12:27:47| 18.5|       1|        18.5|2020-04-29|   2020-04|
|2020-04-29 13:31:33|74.05|       1|       74.05|2020-04-29|   2020-04|
|2020-04-29 16:55:02|23.36|       1|       23.36|2020-04-29|   2020-04|
|2020-04-30 04:04:48| 0.23|       1|      

## 7. Chuẩn hóa chuỗi (String Processing)

In [7]:
# ── Viết thường + trim ────────────────────────────────────────────────────────
df_str = (df_typed
    .withColumn("brand",
                F.trim(F.lower(F.col("brand"))))
    .withColumn("category",
                F.trim(F.lower(F.col("category_code"))))
    # Lấy category cấp 1 (vd: "electronics.tablet" → "electronics")
    .withColumn("category_level1",
                F.split(F.col("category"), r"\.").getItem(0))
    # Lọc giá trị âm hoặc bằng 0
    .filter(F.col("price") > 0)
)

print("Top 10 brand:")
df_str.groupBy("brand").count().orderBy(F.desc("count")).show(10)

print("\nTop 10 category_level1:")
df_str.groupBy("category_level1").count().orderBy(F.desc("count")).show(10)


Top 10 brand:
+-------+-----+
|  brand|count|
+-------+-----+
|samsung|96123|
|  apple|36059|
|unknown|27175|
|    ava|26090|
|  tefal|17945|
|     lg|16674|
| xiaomi|14856|
|philips|12133|
|polaris|10715|
| huawei|10713|
+-------+-----+
only showing top 10 rows


Top 10 category_level1:
+---------------+------+
|category_level1| count|
+---------------+------+
|    electronics|157878|
|     appliances|150338|
|        unknown|129195|
|      computers| 76937|
|      furniture| 24577|
|     stationery|  8877|
|   construction|  3990|
|    accessories|  3098|
|        apparel|  2692|
|           kids|  2302|
+---------------+------+
only showing top 10 rows



## 8. Phân tích khám phá (EDA) - Biểu đồ sau khi xử lý

In [8]:
# ── Lấy mẫu để vẽ biểu đồ ──────────────────────────────────────────────────
df_pd = df_str.sample(False, 0.05, seed=42).toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("EDA - Dữ liệu SAU khi xử lý", fontsize=15, fontweight="bold")

# 1. Phân phối giá
ax = axes[0, 0]
df_pd["price"].clip(upper=df_pd["price"].quantile(0.99)).hist(
    bins=60, ax=ax, color="#3498db", edgecolor="white")
ax.set_title("Phân phối giá sản phẩm (cắt 99th percentile)")
ax.set_xlabel("Giá (USD)")
ax.set_ylabel("Tần suất")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# 2. Top 10 brand theo số giao dịch
ax = axes[0, 1]
brand_cnt = df_pd["brand"].value_counts().head(10)
brand_cnt.plot(kind="barh", ax=ax, color="#e67e22")
ax.set_title("Top 10 Brand theo số giao dịch (mẫu 5%)")
ax.set_xlabel("Số giao dịch")
ax.invert_yaxis()

# 3. Giao dịch theo tháng
ax = axes[1, 0]
monthly = (df_pd.groupby("year_month")["total_amount"]
           .sum().reset_index()
           .sort_values("year_month"))
ax.bar(monthly["year_month"], monthly["total_amount"], color="#9b59b6")
ax.set_title("Doanh thu theo tháng (mẫu 5%)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Total Amount (USD)")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# 4. Top category_level1
ax = axes[1, 1]
cat_cnt = df_pd["category_level1"].value_counts().head(10)
wedges, texts, autotexts = ax.pie(cat_cnt.values,
                                   labels=cat_cnt.index,
                                   autopct="%1.1f%%",
                                   startangle=140,
                                   colors=plt.cm.Set3.colors)
ax.set_title("Phân bổ Category cấp 1 (mẫu 5%)")

plt.tight_layout()
plt.savefig("eda_after_processing.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: eda_after_processing.png")


Đã lưu: eda_after_processing.png


## 9. Spark ML Pipeline

Thực hiện 3 bước:
1. **StringIndexer** – mã hóa `brand` và `category` từ string → số  
2. **VectorAssembler** – gom các cột số thành một vector feature  
3. **StandardScaler** – chuẩn hóa về mean=0, std=1

In [9]:
# ── 9.1  StringIndexer ────────────────────────────────────────────────────────
brand_indexer    = StringIndexer(inputCol="brand",    outputCol="brand_idx",
                                  handleInvalid="keep")
category_indexer = StringIndexer(inputCol="category", outputCol="category_idx",
                                  handleInvalid="keep")

# ── 9.2  VectorAssembler ──────────────────────────────────────────────────────
feature_cols = ["price", "total_amount", "quantity",
                "brand_idx", "category_idx"]

assembler = VectorAssembler(inputCols=feature_cols,
                             outputCol="features_raw",
                             handleInvalid="keep")

# ── 9.3  StandardScaler ───────────────────────────────────────────────────────
scaler = StandardScaler(inputCol="features_raw",
                        outputCol="features_scaled",
                        withMean=True, withStd=True)

# ── Ghép thành Pipeline ───────────────────────────────────────────────────────
pipeline = Pipeline(stages=[brand_indexer,
                             category_indexer,
                             assembler,
                             scaler])

print("Đang fit Pipeline (StringIndexer + VectorAssembler + StandardScaler)...")
pipeline_model = pipeline.fit(df_str)
df_processed   = pipeline_model.transform(df_str)

print("Pipeline hoàn thành!")
df_processed.select("user_id", "event_time", "price",
                     "brand_idx", "category_idx",
                     "features_scaled").show(5, truncate=True)
print(f"Tổng số dòng sau Pipeline: {df_processed.count():,}")


Đang fit Pipeline (StringIndexer + VectorAssembler + StandardScaler)...
Pipeline hoàn thành!
+-------------------+-------------------+------+---------+------------+--------------------+
|            user_id|         event_time| price|brand_idx|category_idx|     features_scaled|
+-------------------+-------------------+------+---------+------------+--------------------+
|1515915625479405878|1970-01-01 00:33:40|  26.6|      8.0|        37.0|[-0.5966576565430...|
|1515915625443334526|1970-01-01 00:33:40|  1.83|     30.0|        23.0|[-0.6779778839747...|
|1515915625479405884|1970-01-01 00:33:40| 92.57|      7.0|         0.0|[-0.3800773011262...|
|1515915625452928840|1970-01-01 00:33:40|127.29|      9.0|         1.0|[-0.2660910961835...|
|1515915625479405912|1970-01-01 00:33:40|  97.2|      0.0|        14.0|[-0.3648769517920...|
+-------------------+-------------------+------+---------+------------+--------------------+
only showing top 5 rows

Tổng số dòng sau Pipeline: 563,456


## 10. So sánh thống kê trước/sau xử lý

In [10]:
print("=== Thống kê tổng hợp ===")
df_processed.select("price", "total_amount", "quantity").describe().show()

# Biểu đồ so sánh số dòng qua các bước
fig, ax = plt.subplots(figsize=(10, 5))
stages = ["Raw", "Sau null-drop", "Sau dedup", "Sau filter price>0"]
counts = [total_raw, cnt_before_dedup, cnt_after_dedup,
          df_processed.count()]
colors_bar = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71"]
bars = ax.bar(stages, counts, color=colors_bar, width=0.5)
ax.set_title("Số dòng qua từng bước xử lý", fontsize=13, fontweight="bold")
ax.set_ylabel("Số giao dịch")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(counts)*0.01,
            f"{val:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("pipeline_stage_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Đã lưu: pipeline_stage_comparison.png")


=== Thống kê tổng hợp ===
+-------+------------------+------------------+--------+
|summary|             price|      total_amount|quantity|
+-------+------------------+------------------+--------+
|  count|            563456|            563456|  563456|
|   mean| 208.3408856237463| 208.3408856237463|     1.0|
| stddev|304.59826272357725|304.59826272357725|     0.0|
|    min|              0.02|              0.02|       1|
|    max|          18328.68|          18328.68|       1|
+-------+------------------+------------------+--------+

Đã lưu: pipeline_stage_comparison.png


## 11. Lưu dữ liệu đã xử lý (Parquet)

In [11]:
# ── Chọn cột cần lưu ─────────────────────────────────────────────────────────
COLS_TO_SAVE = [
    "user_id", "order_id", "product_id",
    "event_time", "date", "year_month",
    "brand", "brand_idx",
    "category", "category_level1", "category_idx",
    "price", "quantity", "total_amount",
    "features_scaled"
]

df_final = df_processed.select(*COLS_TO_SAVE)

# ── Ghi Parquet phân vùng theo year_month ────────────────────────────────────
(df_final
 .write
 .mode("overwrite")
 .partitionBy("year_month")
 .parquet(OUTPUT_PATH))

print(f" Đã lưu dữ liệu tại: {OUTPUT_PATH}")
print(f"   Tổng số dòng       : {df_final.count():,}")
print(f"   Số cột             : {len(COLS_TO_SAVE)}")

# Đọc lại để xác nhận
df_verify = spark.read.parquet(OUTPUT_PATH)
print(f"\n Xác nhận đọc lại: {df_verify.count():,} dòng")
df_verify.printSchema()


 Đã lưu dữ liệu tại: /home/jovyan/work/data/cleaned_orders.parquet
   Tổng số dòng       : 563,456
   Số cột             : 15

 Xác nhận đọc lại: 563,456 dòng
root
 |-- user_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- brand_idx: double (nullable = true)
 |-- category: string (nullable = true)
 |-- category_level1: string (nullable = true)
 |-- category_idx: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- features_scaled: vector (nullable = true)
 |-- year_month: string (nullable = true)



## 12. Tóm tắt & Kết thúc

In [12]:
print("=" * 60)
print("        TỔNG KẾT QUÁ TRÌNH TIỀN XỬ LÝ")
print("=" * 60)
print(f"  Dữ liệu gốc (raw)          : {total_raw:>12,} dòng")
print(f"  Sau xử lý null             : {cnt_before_dedup:>12,} dòng")
print(f"  Sau xử lý duplicate        : {cnt_after_dedup:>12,} dòng")
print(f"  Sau lọc giá và ML Pipeline : {df_final.count():>12,} dòng")
print(f"  Số cột đầu ra              : {len(COLS_TO_SAVE):>12}")
print(f"  Output format              : {'Parquet (partitioned by year_month)':>12}")
print("=" * 60)
print("\nCác file biểu đồ:")
print("  - missing_before.png")
print("  - dedup_comparison.png")
print("  - eda_after_processing.png")
print("  - pipeline_stage_comparison.png")

spark.stop()
print("\n SparkSession đã dừng.")


        TỔNG KẾT QUÁ TRÌNH TIỀN XỬ LÝ
  Dữ liệu gốc (raw)          :    2,633,521 dòng
  Sau xử lý null             :      564,169 dòng
  Sau xử lý duplicate        :      563,495 dòng
  Sau lọc giá và ML Pipeline :      563,456 dòng
  Số cột đầu ra              :           15
  Output format              : Parquet (partitioned by year_month)

Các file biểu đồ:
  - missing_before.png
  - dedup_comparison.png
  - eda_after_processing.png
  - pipeline_stage_comparison.png

 SparkSession đã dừng.
